In [ ]:
import pandas as pd
import geopandas as gpd
from urllib.parse import quote

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhe-CUNY/python-pandas-geopandas/blob/main/02_geopandas.ipynb) 02_geopandas.ipynb

In [ ]:
# show complaints in cb
where_clause = f"created_date>='2026-06-01T00:00:00' AND community_board='05 MANHATTAN'"

url = (
    f"https://data.cityofnewyork.us/resource/erm2-nwe9.csv"
    f"?$limit=10000"
    f"&$where={quote(where_clause)}"
)

df = pd.read_csv(url)

In [ ]:
df

In [ ]:
# convert to geopandas

gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df.longitude, df.latitude), crs="EPSG:4326"
)

In [ ]:
gdf[['unique_key','geometry']]

In [ ]:
gdf.plot()

In [ ]:
gdf[:10].explore()

### let's style the plot

In [ ]:
gdf.plot(
    color="red",
    linewidth=0.1,
    alpha=0.1,
    figsize=(10, 6)
)

#### add more data and context
https://boundaries.beta.nyc/?map=bid

https://data.cityofnewyork.us/Business/Business-Improvement-Districts/ejxk-d93y


In [ ]:
# get the bid data
bids = gpd.read_file('https://data.cityofnewyork.us/api/v3/views/7jdm-inj8/query.geojson')

In [ ]:
# plot more two layers together

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 8))

bids[bids['f_all_bi_1'] == 'Manhattan'].plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)

gdf.plot(ax=ax, color="red", markersize=1, alpha=0.1)

ax.set_title("Complaints and bids")
ax.set_axis_off()
plt.show()

# a bit messy, lets filter down

In [ ]:
gpd.sjoin(gdf, bids, how = 'left')

In [ ]:
complaints_with_bids = gdf.sjoin(bids, how = 'left')

In [ ]:
complaints_with_bids[pd.notna(complaints_with_bids['f_all_bi_2'])][['f_all_bi_1','f_all_bi_2','geometry']].explore()

In [ ]:
complaints_with_bids['geometry'] = complaints_with_bids.to_crs(2263).buffer(100)

In [ ]:
complaints_with_bids[['complaint_type','geometry']].sjoin(bids.to_crs(2263), how = 'left')['f_all_bi_2'].value_counts()

In [ ]:
gdf.sjoin(bids, how = 'left')

In [ ]:
# which bid has the most complaints

### let's join the spatial join the data

### Another example

https://data.cityofnewyork.us/City-Government/Historical-Voter-Turnout/rixx-fc37/about_data

https://www.nyc.gov/content/planning/pages/resources/datasets/state-assembly

https://data.cityofnewyork.us/City-Government/Election-Districts/h2n3-98hq

In [ ]:
# get the turnout data
df = pd.read_csv('https://data.cityofnewyork.us/api/v3/views/rixx-fc37/query.csv')

In [ ]:
# filter for only council districts
df2 = df[df['geo_level'] == 'Council District']

# pivot to get the avg turnout by district 
df3 = pd.pivot_table(
    df2,
    values='turnout',
    index='geo_unit',
    aggfunc='mean'
).sort_values(by = 'turnout').reset_index()

In [ ]:
# get the district geoms
district = gpd.read_file('https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_City_Council_Districts/FeatureServer/0/query?where=1=1&outFields=*&outSR=4326&f=pgeojson')

In [ ]:
# convert the CounDist to a str so they keys match on both sides
district['CounDist'] = district['CounDist'].astype(str)

In [ ]:
# do a table join on CounDist 1 .. 51 --> geo_unit 1 -- 51 
# then plot on the map
district.merge(df3, how = 'left', left_on = 'CounDist', right_on = 'geo_unit').explore(
    column = 'turnout', cmap = 'magma'
)